# Семинар 08. Исключения и контекстные менеджеры

Пять задач о том, что делать после ошибки: прервать операцию, подставить значение, продолжить обработку или освободить ресурс. У каждой задачи своё правило; одинаковый `except` на всё здесь не подойдёт.

## Цели

- проверять условия операции до изменения состояния;
- ловить только ожидаемые ошибки в подходящем месте;
- сохранять исключение и выполнять обязательные действия через `finally`;
- составлять отчёт об отклонённых записях;
- закрывать текстовый поток при успехе и при ошибке.

## Перед началом

Нужны функции, строки, списки, словари и JSON из первых семи занятий, а также [лекция 8](lecture.ipynb). Классы, декораторы и генераторы не используются.

Задания 1–3 — небольшие функции, задания 4–5 — самостоятельная обработка данных. Если уверенно чувствуете себя с исключениями, можно начать с четвёртого.

В заготовках стоит `pass`, а вызовы `check_task_N()` закомментированы. Сначала напишите функцию, затем выполните ячейку с определением проверки и уберите `#` перед её вызовом. `Run All` исходного ноутбука проходит, но это не означает, что задачи решены: проверки запускаются отдельно. Готовых решений в ноутбуке нет.

## Как проверить ожидаемое исключение

Проверка должна упасть, если исключение не возникло. Для этого удобно использовать `else` у `try`: оно выполнится после успешного вызова. В примере ниже проверяется встроенная операция, это не решение одной из задач.

In [ ]:
try:
    int("не число")
except ValueError:
    pass
else:
    raise AssertionError("Ожидался ValueError")

## Задание 1. Списание со счёта

Напишите `withdraw(balance, amount)`: верните остаток после списания.

- `balance` и `amount` — целые числа;
- отрицательный остаток до операции, неположительная сумма или нехватка денег — `ValueError`;
- списание ровно всего остатка допустимо;
- ошибка не должна превращаться в нулевой остаток.

Примеры: `withdraw(100, 30) == 70`, `withdraw(100, 100) == 0`; для `(100, 101)` нужен `ValueError`.

Ограничения: числа по модулю не больше 10⁹. Требуется `O(1)` времени и памяти.

In [ ]:
def withdraw(balance: int, amount: int) -> int:
    pass

In [ ]:
def check_task_1():
    assert withdraw(100, 30) == 70
    assert withdraw(100, 100) == 0
    assert withdraw(1, 1) == 0
    for balance, amount in [(100, 101), (100, 0), (100, -1), (-1, 1), (0, 1)]:
        try:
            withdraw(balance, amount)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Ожидался ValueError для {(balance, amount)}")
    print("Задание 1: проверки пройдены")

# check_task_1()

## Задание 2. Запасное значение

Напишите `read_limit(text, default)`: попробуйте получить целое число по правилам `int(text)`.

- Если текст не разбирается как целое число, верните `default`.
- Если число разобралось, но оно отрицательное, поднимите `ValueError`.
- Если число неотрицательное, верните его, включая 0.

`default` гарантированно неотрицательное целое число. Тип `text` — `str`, длина до 100 символов; разобранное число по модулю не больше 10⁹.

Примеры: `read_limit("15", 10) == 15`, `read_limit("много", 10) == 10`; `read_limit("-1", 10)` должен поднять `ValueError`, а не вернуть 10.

Критерий: ошибка преобразования и недопустимое значение обрабатываются по разным правилам. Время — `O(len(text))`.

In [ ]:
def read_limit(text: str, default: int) -> int:
    pass

In [ ]:
def check_task_2():
    assert read_limit("15", 10) == 15
    assert read_limit("  +007 ", 10) == 7
    assert read_limit("0", 10) == 0
    assert read_limit("много", 10) == 10
    assert read_limit("", 0) == 0
    assert read_limit("1.5", 7) == 7
    try:
        read_limit("-1", 10)
    except ValueError:
        pass
    else:
        raise AssertionError("Отрицательный лимит нельзя подменить запасным")
    print("Задание 2: проверки пройдены")

# check_task_2()

## Задание 3. Журнал разбора

Напишите `decode_logged(text, events)`. Перед разбором JSON через `json.loads()` допишите в переданный список строку `"start"`. При успехе допишите `"ok"` и верните разобранное значение. Перед любым выходом допишите `"finish"`, используя `finally`.

Ошибка `json.JSONDecodeError` должна выйти наружу без подмены. Прежние записи журнала сохраняются. Корректные значения `null`, `false`, `0` не являются ошибкой.

Пример успеха: `events == ["start", "ok", "finish"]`. При некорректном JSON: `events == ["start", "finish"]` и вызов завершается исключением.

Типы аргументов соответствуют аннотациям, длина строки до 100 000 символов, глубина JSON до 20. Кроме результата разбора, создаются только записи журнала.

In [ ]:
import json


def decode_logged(text: str, events: list[str]) -> object:
    pass

In [ ]:
def check_task_3():
    events = ["earlier"]
    assert decode_logged('{"count": 0}', events) == {"count": 0}
    assert events == ["earlier", "start", "ok", "finish"]
    for text, expected in [("null", None), ("false", False), ("0", 0)]:
        events = []
        result = decode_logged(text, events)
        assert result == expected
        assert type(result) is type(expected)
        assert events == ["start", "ok", "finish"]
    events = []
    broken = '{"count":}'
    try:
        decode_logged(broken, events)
    except json.JSONDecodeError as error:
        assert error.doc == broken
    else:
        raise AssertionError("Ошибка JSON была подавлена")
    assert events == ["start", "finish"]
    print("Задание 3: проверки пройдены")

# check_task_3()

## Задание 4. Пакет заявок на списание

Напишите `process_requests(initial, requests)`. Каждая строка списка — сумма одного списания. Строка разбирается по правилам `int()`. Выполните допустимые списания в исходном порядке и верните `(остаток, номера_отклонённых_заявок)`.

Заявка отклоняется, если сумма не разбирается, не положительна или на счёте недостаточно денег. Отклонённая заявка не меняет остаток и не мешает обработать следующие. Номера начинаются с 1. Пустая строка — ошибочная заявка.

Начальный остаток гарантированно неотрицательный. Суммы по модулю не превышают 10⁹; не более 100 000 строк длиной до 100 символов. Входной список не меняется.

Пример: при остатке 100 заявки `["30", "80", "-2", "ошибка", "70"]` дают `(0, [2, 3, 4])`.

Критерии: сохранён порядок, неудачная операция не оставляет изменений, перехватываются только ожидаемые ошибки. Время — `O(L)` по суммарной длине строк, дополнительная память — список отклонений.

In [ ]:
def process_requests(
    initial: int, requests: list[str]
) -> tuple[int, list[int]]:
    pass

In [ ]:
def check_task_4():
    assert process_requests(100, ["30", "80", "-2", "ошибка", "70"]) == (0, [2, 3, 4])
    assert process_requests(0, []) == (0, [])
    assert process_requests(0, ["1", "0", "-1", ""]) == (0, [1, 2, 3, 4])
    assert process_requests(10, ["10", "1"]) == (0, [2])
    assert process_requests(5, ["6", "5"]) == (0, [1])
    requests = ["  +003 ", "не число", "2"]
    assert process_requests(5, requests) == (0, [2])
    assert requests == ["  +003 ", "не число", "2"]
    print("Задание 4: проверки пройдены")

# check_task_4()

## Задание 5. Среднее из открытого отчёта

Напишите `read_mean(source)`. Функция получает открытый текстовый поток: обычный файл или `StringIO`. Прочитайте целые неотрицательные числа, по одному в строке; строки, пустые после удаления пробелов, пропускаются. Верните среднее арифметическое как `float`.

- Нечисловая или отрицательная значимая строка — `ValueError`.
- Если чисел нет, поднимите `ValueError`.
- Ошибка чтения, например `OSError`, должна выйти наружу без подмены.
- Функция принимает владение потоком: **после любого выхода он закрыт**. Используйте `with`.
- Проверка и вычисление должны выполняться внутри времени жизни открытого потока.

Для локальной проверки используется `StringIO`: он хранит текст в памяти, но поддерживает чтение, `with` и свойство `.closed`, как файл. Внешние файлы для задания не нужны.

Не более 100 000 строк длиной до 100 символов; разобранные числа по модулю не больше 10⁹. Критерии: правильное среднее и политика ошибок, закрытие при успехе и ошибке. Время — `O(L)` по длине текста; дополнительная память — `O(1)` без учёта текущей строки.

In [ ]:
from io import StringIO
from typing import TextIO


def read_mean(source: TextIO) -> float:
    pass

In [ ]:
def check_task_5():
    source = StringIO("10\n\n 20 \n30\n")
    result = read_mean(source)
    assert result == 20.0
    assert type(result) is float
    assert source.closed

    source = StringIO("0\n")
    assert read_mean(source) == 0.0
    assert source.closed

    for text in ["", " \n\t\n", "10\nошибка\n20\n", "10\n-1\n"]:
        source = StringIO(text)
        try:
            read_mean(source)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Ожидался ValueError для {text!r}")
        assert source.closed
    print("Задание 5: проверки пройдены")

# check_task_5()

## Обсуждение

1. Почему нельзя ловить любой `ValueError` вокруг всей программы и считать, что это всегда ошибка пользователя?
2. Почему `finally` с `return` может испортить правильную обработку ошибки?
3. Чем отличается отклонить отдельную заявку от прервать чтение всего отчёта?
4. Если файл закрыт после ошибки записи, означает ли это, что прежнее содержимое восстановлено?
5. Кто должен закрывать поток, переданный функции? Как вызывающий узнает об этом правиле?

В заданиях с `source` владение явно передано функции. В другой программе договор может быть иным: само наличие параметра-файла ещё не означает, что функция вправе его закрыть.

## Домашняя работа

> **Домашняя работа.** В материалах курса приведены **предполагаемые списки домашних задач**. Домашние задания нужно решать и сдавать в **Яндекс.Контесте**. Актуальный список задач, условия и требования к оформлению находятся в Контесте. **При расхождениях информация в Контесте имеет приоритет.** Ссылки на контесты публикуются на **вики курса**.

Условия находятся в [tasks.md](tasks.md). Три небольшие задачи закрепляют правила исключений, две более крупные — обработку команд и чтение проверяемого JSON.

## Итоги

У каждой операции есть конкретный договор: какие данные допустимы, какая ошибка ожидается и что должно остаться после неудачи. `except` выбирает способ восстановления, `finally` выполняет обязательные действия, а `with` ограничивает время жизни ресурса. Ошибка программы не должна незаметно превращаться в пропущенную строку или запасное значение.